# conv-stride-downsample — ex2: find padding that gives a clean stride-S halve

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-stride-downsample`. Running the final beacon cell reports progress against the `CNN: Stride downsample arithmetic` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Stride downsample arithmetic` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-stride-downsample`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-stride-downsample"
DD_SUBTOPIC = "CNN: Stride downsample arithmetic"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Same-padding for clean stride-S halving — quick refresher

Default `Conv2d(stride=S, K)` (no padding) gives `H_out = (H_in - K) // S + 1` — usually 1 short of the 'clean halve' answer `H_in // S`. The fix is padding `P` on each side, which generalizes the formula to:

```
H_out = (H_in + 2*P - K) // S + 1
```

**The 'clean half' goal.** For `S = 2` you want `H_out == H_in // 2`. Plug in and solve: `H_in // 2 = (H_in + 2P - K) // 2 + 1`, which gives `P = (K - 1) // 2` for odd `K` (and `H_in` even).

**The general 'same-pad for stride-S' rule.** For odd kernel size `K`:

```
P = (K - 1) // 2
```

This works for `K = 3 → P = 1`, `K = 5 → P = 2`, `K = 7 → P = 3`. With this padding, every stride-`S` conv divides the input length cleanly by `S` (assuming the input is itself divisible by `S`).

**Why this matters.** ResNet, U-Net, and every modern CNN family uses this exact pairing — odd kernel + `(K-1)//2` padding + stride-2 — to get predictable 2× downsampling per stage. The off-by-one without padding would compound across 4-5 stages and produce ugly non-power-of-2 sizes.

### Exercise 2 — find padding that gives a clean stride-S halve

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the `(H + 2P - K) // S + 1` shape formula to derive the padding `P = (K - 1) // 2` that makes an odd-kernel stride-S conv output equal `H_in // S` exactly.
> Keywords: stride, padding, same-padding, downsample, off-by-one
> ```

**KCs targeted:** `same-pad-formula-odd-kernel`, `stride-pad-conv-shape`

Implement `ex2_same_pad_for_halve(k)`. Given an odd kernel size `k`, return the integer padding `P` such that a `Conv2d(..., kernel_size=k, stride=2, padding=P)` applied to an input of even spatial size `H_in` produces output of size exactly `H_in // 2`.

**Derivation.** The general output formula is:

```
H_out = (H_in + 2*P - K) // S + 1
```

For `S = 2`, set `H_out = H_in // 2`. With odd `K` and even `H_in`, the closed-form padding is:

```
P = (K - 1) // 2
```

**Constraint.** Your implementation must validate that `k` is ODD — raise `ValueError("kernel must be odd")` if `k` is even, since the closed form breaks down for even kernels (you'd need asymmetric padding).

**Hint.** The formula is genuinely just `(k - 1) // 2`. The work is in convincing yourself by working through `k = 3, 5, 7`: each should produce `P = 1, 2, 3` respectively, and the test verifies the *actual* conv output shape equals `H_in // 2` for several `H_in` values.

The test cross-checks the predicted padding against real `nn.Conv2d` output shapes for multiple `(K, H_in)` combos.

In [ ]:
def ex2_same_pad_for_halve(k: int) -> int:
    if k % 2 == 0:
        raise ValueError(f'kernel must be odd, got k={k}')
    return (k - 1) // 2


<details><summary>Solution</summary>

```python
def ex2_same_pad_for_halve(k: int) -> int:
    if k % 2 == 0:
        raise ValueError(f'kernel must be odd, got k={k}')
    return (k - 1) // 2
```

**Why `(k - 1) // 2`.** Plug into the shape formula:
```
H_out = (H_in + 2*((k-1)//2) - k) // 2 + 1
      = (H_in + (k-1) - k) // 2 + 1     # for odd k, 2 * (k-1)/2 = k-1
      = (H_in - 1) // 2 + 1
      = H_in // 2                       # for even H_in
```
The off-by-one is absorbed by the `+ 1` from the leading-window term; padding adds exactly enough on each side to make the math work.

**Why the odd-K constraint.** For even K, `(k - 1) / 2` is not an integer. You'd need asymmetric padding (`k // 2` on one side, `k // 2 - 1` on the other) — but `nn.Conv2d(padding=...)` only accepts symmetric padding. Even kernels require `F.pad` first, then `Conv2d(padding=0)`. The constraint sidesteps this.

**Generalizing to stride S.** The same derivation gives `P = (k - 1) // 2` for ANY odd k and ANY stride S, provided `H_in % S == 0`. The padding formula is independent of stride — stride determines the output spacing, padding determines the fence-post offset.

**Real-world use.** Every modern CNN's stride-2 downsample layer uses this padding. Look at any ResNet block's `nn.Conv2d` constructor and you'll see `kernel_size=3, stride=2, padding=1` — that's this drill's answer applied.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()